In [1]:
import matplotlib.pyplot as plt
import pandas as pd


import atlas.eugene as eug

In [2]:
import black
import jupyter_black

jupyter_black.load(
    # lab=False,
    line_length=78,
    # verbosity="DEBUG",
    target_version=black.TargetVersion.PY310,
)

# Historic Downtown Eugene

## Data Sources

* [Eugene_Neighborhoods_-_HUB.geojson](https://mapping.eugene-or.gov/maps/9e55392992c142b8a1c51a33d9291d8e)
* [Eugene_Taxlots_-_HUB.geojson](https://mapping.eugene-or.gov/maps/9e7b804394f048368415b840c5b6f9a1)
* [lane-county-or-tax-and-jail-scrape](https://github.com/nbirnel/lane-county-or-tax-and-jail-scrape)
  - account_tax_payer_lot.csv: maptaxlot, acreage, account
  - assessments.csv: account, assessments
  - commercial_improvements.csv: commercial (including multi-residential) buildings
  - residential_buildings.csv: single-family housing



In [3]:
assessed_gdf = eug.get_assessed_gdf()
single_family_housing = eug.get_single_family_housing_gdf()
multi_family_housing = eug.get_multi_family_housing_gdf()
non_residential_buildings = eug.get_non_residential_gdf()
neighborhoods = eug.get_neighborhood_gdf()

In [4]:
columns = [
    "maptaxlot",
    "geometry",
    "situs_address",
    "floor_number",
    "acreage",
    "neighborhood",
    "ward",
    "Year Built",
    "Decade Built",
    "description",
]

sfh = single_family_housing[columns].copy()
sfh["class"] = "Single Family Housing"

mfh = multi_family_housing[columns].copy()
mfh["class"] = "Multi Family Housing"
nrb = non_residential_buildings[columns].copy()
nrb["class"] = "Non-Residential"
buildings = pd.concat([sfh, mfh, nrb])
improvement_types = pd.CategoricalDtype(
    categories=[
        "Single Family Housing",
        "Multi Family Housing",
        "Non-Residential",
    ],
    ordered=True,
)
buildings["class"] = buildings["class"].astype(improvement_types)

In [5]:
city_core = [
    "Downtown Neighborhood Association",
    "Whiteaker Community Council",
    "Chambers Westside Neighbors",
    "Jefferson Westside Neighbors",
    "Friendly Area Neighbors",
    "Amazon Neighbors Association",
    "Fairmount Neighbors",
    "South University Neighborhood Association",
    "West University Neighbors",
]

In [6]:
downtown_old_houses = single_family_housing[
    (single_family_housing["neighborhood_abbreviation"] == "DNA")
    & (single_family_housing.year_built < 1941)
].sort_values(by="year_built")


# center = [
#    gdf.geometry.union_all().centroid.y,
#    gdf.geometry.union_all().centroid.x,
# ]
m = eug.explore(
    downtown_old_houses,
    column="Decade Built",
    title=f"Downtown Eugene: Pre-War Houses",
    cmap="OrRd_r",
    legend=True,
    tooltip=[
        "situs_address",
        "neighborhood",
        "Year Built",
        "Acreage",
    ],
    zoom_start=15,
)

m = neighborhoods[
    neighborhoods.Neighborhood == "Downtown Neighborhood Association"
].explore(m=m, fill=False, color="grey", tooltip=False)
m

In [7]:
old_apartments = multi_family_housing[
    multi_family_housing.year_built < 1941
].sort_values(by="year_built")

In [8]:
dna_old_apartments = old_apartments[
    old_apartments.neighborhood_abbreviation == "DNA"
].sort_values(by="year_built")

m = eug.explore(
    dna_old_apartments,
    column="Decade Built",
    legend=True,
    title=f"Downtown Eugene: Pre-War Apartments",
    cmap="OrRd_r",
    tooltip=[
        "situs_address",
        "maptaxlot",
        "Year Built",
        "Acreage",
        "description",
        "floor_number",
        "effective_year_built",
        "sq_ft",
    ],
    zoom_start=15,
)
m = neighborhoods[
    neighborhoods.Neighborhood == "Downtown Neighborhood Association"
].explore(m=m, fill=False, color="grey", tooltip=False)
m

In [9]:
commercial = eug.get_commercial_buildings_gdf()

In [10]:
old_commercial = commercial[commercial.year_built < 1941].sort_values(
    by="year_built"
)

In [11]:
dna_old_commercial = old_commercial[
    old_commercial.neighborhood_abbreviation == "DNA"
]

In [12]:
m = eug.explore(
    dna_old_commercial,
    column="Decade Built",
    legend=True,
    title=f"Downtown Eugene: Pre-War Commercial Buildings",
    cmap="OrRd_r",
    tooltip=[
        "situs_address",
        "maptaxlot",
        "Year Built",
        "Acreage",
        "description",
        "floor_number",
        "effective_year_built",
        "sq_ft",
    ],
    zoom_start=15,
)
m = neighborhoods[
    neighborhoods.Neighborhood == "Downtown Neighborhood Association"
].explore(m=m, fill=False, color="grey", tooltip=False)
m